# 02 · Train the EfficientNet-B0 classifier

Fine-tunes an ImageNet-pretrained EfficientNet-B0 (single real/fake logit) with mixed precision, OneCycle LR, JPEG-compression and random-erasing augmentation, early stopping on validation AUC, and a decision threshold tuned on the validation set.

**Notebook settings (right sidebar)**
- **Accelerator:** GPU T4 x2 (not P100)
- **Internet:** On
- **Inputs:** output of **01_extract_faces**

**Produces:** `/kaggle/working/outputs/checkpoints/best.pt` and `history.json`. When it finishes, use **Save Version → Save & Run All**, then add this notebook's output as an input to the next notebook.

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Uses the repo if it is attached as an input (zip uploaded as a Kaggle dataset, or an earlier
# notebook's output); otherwise clones it. For a private repo add a GITHUB_TOKEN secret.
import os, sys, glob, shutil, subprocess

PROJECT = "/kaggle/working/project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
    if found:
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))
    else:
        token = ""
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            pass
        url = REPO.replace("https://", f"https://{token}@") if token else REPO
        r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
        if r.returncode:
            raise RuntimeError("git clone failed: attach the project as a dataset or add a GITHUB_TOKEN secret")
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
print("project ready:", sorted(os.listdir(PROJECT)))

In [ ]:
from src.kaggle_utils import setup
setup(data=True)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## Settings
`SUBSET` trains on a random subset of frames for a quicker run (`None` = all).

In [ ]:
EPOCHS, BATCH, LR, PATIENCE, SUBSET = 8, 32, 3e-4, 3, None
WEIGHT_DECAY = 0.05   # higher = stronger regularisation against overfitting

In [ ]:
args = f"--epochs {EPOCHS} --batch-size {BATCH} --lr {LR} --patience {PATIENCE} --weight-decay {WEIGHT_DECAY} --workers 4"
if SUBSET: args += f" --subset {SUBSET}"
!python -m src.classifier.train {args}

## Training curves

In [ ]:
h = pd.read_json("/kaggle/working/outputs/checkpoints/history.json")
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.5))
a.plot(h.epoch, h.train_loss, marker="o"); a.set_title("train loss"); a.set_xlabel("epoch")
b.plot(h.epoch, h.val_auc, marker="o", label="val AUC"); b.plot(h.epoch, h.val_acc, marker="o", label="val acc @0.5")
if "val_acc_tuned" in h: b.plot(h.epoch, h.val_acc_tuned, marker="o", label="val acc @tuned threshold")
b.set_title("validation"); b.set_xlabel("epoch"); b.legend()
plt.tight_layout(); plt.show()
display(h.round(4))